# 03. Retriever Benchmarking for RAG

This notebook benchmarks retrieval methods in a tiny RAG-style setup.

The most important point in this notebook is explicit:

> A weak retriever limits downstream answer quality, no matter how fluent the generator is.

We will compare:
- TF-IDF lexical retrieval
- BM25 lexical retrieval
- dense retrieval with sentence embeddings
- FAISS vector search over the same dense embeddings
- a simple hybrid score


The workshop treats the system as retrieval-first:
- tokenization defines what the model sees
- retrieval quality often controls answer quality
- generators cannot reliably compensate for weak retrieval
- evaluation should include groundedness, citation, abstention, bilingual behavior, and community-review placeholders

Current notebook: `03_retriever_benchmarking_for_rag.ipynb`

This notebook benchmarks retrievers directly, because weak retrieval is often the main reason a RAG assistant gives a poor or unsupported answer.

This is a core workshop notebook.

Workshop sequence:
1. `01_tokenization_playground.ipynb`
2. `02_embeddings_and_similarity.ipynb`
3. `03_retriever_benchmarking_for_rag.ipynb`
4. `04_llm_comparison_in_same_rag_pipeline.ipynb`
5. `05_evaluating_rag_systems.ipynb`
6. `06_optional_lora_or_domain_adaptation.ipynb` (optional)


## Quick concept refresher

- Tokenization turns text into units that models can process.
- Retrieval chooses which passages are available as evidence.
- The retriever selects context; the generator turns context into an answer.
- Evaluation in archive assistants is multi-layered because the system must be useful, grounded, reviewable, and appropriate.


In [ ]:
# Self-contained runtime setup for this notebook.
# Each notebook includes its own seed and device checks so it can run independently in Colab.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")
print(f"Detected device: {DEVICE}")
print(f"Seed set to: {SEED}")

NOTEBOOK_CONTEXT = {
    "seed": SEED,
    "device": DEVICE,
    "notebook": __name__ if "__name__" in globals() else "notebook",
    "framing": "retrieval-first archive assistant workshop",
}

NOTEBOOK_CONTEXT


In [ ]:
# In Colab, uncomment if needed.

# !pip -q install sentence-transformers scikit-learn pandas matplotlib seaborn rank-bm25 faiss-cpu

print("Retriever benchmarking dependencies are ready once the packages are installed.")


In [ ]:
# Imports.

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import faiss


In [ ]:
# A tiny curated corpus with metadata.

archive_passages = [
    {
        "doc_id": "A01",
        "source": "oral_history_box_1",
        "language": "en",
        "type": "oral_history",
        "access_level": "public",
        "text": "An elder describes the river crossing route and explains why some place-names keep older spellings in the catalog.",
    },
    {
        "doc_id": "A02",
        "source": "protocol_memo_2021",
        "language": "en",
        "type": "policy_note",
        "access_level": "restricted_placeholder",
        "text": "Ceremonial recordings require a community review before access is granted, even when short public summaries are available.",
    },
    {
        "doc_id": "A03",
        "source": "winter_story_index",
        "language": "en",
        "type": "story_index",
        "access_level": "seasonal_placeholder",
        "text": "Several teaching stories are marked as winter-only and should not be surfaced as general-purpose examples outside that season.",
    },
    {
        "doc_id": "A04",
        "source": "bilingual_catalog_sheet",
        "language": "fr",
        "type": "catalog_note",
        "access_level": "public",
        "text": "Le catalogue bilingue note des variantes orthographiques, des noms de lieux et les règles de citation pour les résumés publics.",
    },
    {
        "doc_id": "A05",
        "source": "transcription_qc_log",
        "language": "en",
        "type": "quality_log",
        "access_level": "internal_placeholder",
        "text": "The transcript quality log reports OCR errors, merged speaker turns, and missing diacritics in several archived interviews.",
    },
    {
        "doc_id": "A06",
        "source": "access_training_notes",
        "language": "mixed",
        "type": "training_note",
        "access_level": "public",
        "text": "Training notes explain a kinship-based access practice and warn that retrieval should respect governance metadata before showing results.",
    },
    {
        "doc_id": "A07",
        "source": "language_lessons_audio",
        "language": "en",
        "type": "lesson",
        "access_level": "public",
        "text": "A beginner language lesson pairs short audio clips with English glosses and a note about respectful pronunciation practice.",
    },
    {
        "doc_id": "A08",
        "source": "donor_agreement_summary",
        "language": "en",
        "type": "agreement",
        "access_level": "restricted_placeholder",
        "text": "The donor agreement summary states that certain recordings can be described at a high level but not quoted directly in generated answers.",
    },
]

corpus_df = pd.DataFrame(archive_passages)
corpus_df


In [ ]:
# A tiny hand-built retrieval benchmark.
# Each query includes the relevant document ids we hope retrieval will surface.

benchmark_queries = [
    {
        "query_id": "Q01",
        "query": "Which records need community review before access?",
        "relevant_doc_ids": ["A02"],
    },
    {
        "query_id": "Q02",
        "query": "Quels documents parlent des variantes orthographiques et des noms de lieux?",
        "relevant_doc_ids": ["A04", "A01"],
    },
    {
        "query_id": "Q03",
        "query": "Find material that should only be used in winter teaching contexts.",
        "relevant_doc_ids": ["A03"],
    },
    {
        "query_id": "Q04",
        "query": "Which notes mention OCR errors or merged speaker turns?",
        "relevant_doc_ids": ["A05"],
    },
    {
        "query_id": "Q05",
        "query": "Find records about kinship-based access practices and governance metadata.",
        "relevant_doc_ids": ["A06"],
    },
]

benchmark_df = pd.DataFrame(benchmark_queries)
benchmark_df


In [ ]:
# Chunking matters in real systems.
# Here we keep one passage per row, you can experiment with chunk size below.

CHUNK_SIZE_WORDS = 80

print("Current educational chunk size setting:", CHUNK_SIZE_WORDS)
print("In real archives, chunk size and metadata filtering can change retrieval quality a lot.")


In [ ]:
# Build lexical retrievers.
# TF-IDF and BM25 often behave differently when lexical overlap is sparse.

corpus_texts = corpus_df["text"].tolist()

tfidf_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)

bm25_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(bm25_corpus)


In [ ]:
# Build dense retrieval resources.
# We compare brute-force cosine retrieval against FAISS for the same embeddings.

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)

dense_embeddings = embedding_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

faiss_index = faiss.IndexFlatIP(dense_embeddings.shape[1])
faiss_index.add(dense_embeddings)

print("Dense embedding matrix shape:", dense_embeddings.shape)


In [ ]:
# Helper functions for each retriever.

def retrieve_tfidf(query: str, top_k: int = 3):
    query_vector = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix)[0]
    result = corpus_df.copy()
    result["score"] = scores
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


def retrieve_bm25(query: str, top_k: int = 3):
    scores = bm25.get_scores(query.lower().split())
    result = corpus_df.copy()
    result["score"] = scores
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


def retrieve_dense(query: str, top_k: int = 3):
    query_vector = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(query_vector, dense_embeddings)[0]
    result = corpus_df.copy()
    result["score"] = scores
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


def retrieve_faiss(query: str, top_k: int = 3):
    query_vector = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(query_vector, top_k)
    result = corpus_df.iloc[indices[0]].copy()
    result["score"] = scores[0]
    return result.reset_index(drop=True)


def retrieve_hybrid(query: str, top_k: int = 3):
    tfidf_scores = cosine_similarity(tfidf_vectorizer.transform([query]), tfidf_matrix)[0]
    dense_scores = cosine_similarity(
        embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True),
        dense_embeddings,
    )[0]

    combined = 0.5 * tfidf_scores + 0.5 * dense_scores
    result = corpus_df.copy()
    result["score"] = combined
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)


In [ ]:
# Quick qualitative comparison for a single query.

sample_query = benchmark_df.iloc[1]["query"]
print("Sample query:", sample_query)

for name, fn in {
    "tfidf": retrieve_tfidf,
    "bm25": retrieve_bm25,
    "dense": retrieve_dense,
    "faiss": retrieve_faiss,
    "hybrid": retrieve_hybrid,
}.items():
    print(f"\nRetriever: {name}")
    display(fn(sample_query, top_k=3)[["doc_id", "language", "type", "access_level", "score", "text"]])


In [ ]:
# Metric helpers.
# precision_at_k answers: how many of the top-k were relevant?
# hit_rate_at_k answers: did we retrieve at least one relevant item?
# recall_at_k answers: how much of the known relevant set did we recover?

def precision_at_k(retrieved_ids, relevant_ids, k):
    return sum(doc_id in relevant_ids for doc_id in retrieved_ids[:k]) / max(k, 1)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    return float(any(doc_id in relevant_ids for doc_id in retrieved_ids[:k]))


def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    return sum(doc_id in relevant_ids for doc_id in retrieved_ids[:k]) / len(relevant_ids)


In [ ]:
# Benchmark every retriever over the small hand-built evaluation set.
# These results are tiny but still useful for discussion.

retrievers = {
    "tfidf": retrieve_tfidf,
    "bm25": retrieve_bm25,
    "dense": retrieve_dense,
    "faiss": retrieve_faiss,
    "hybrid": retrieve_hybrid,
}

evaluation_rows = []
ranking_rows = []

TOP_K = 3

for record in benchmark_queries:
    query = record["query"]
    relevant_ids = record["relevant_doc_ids"]

    for retriever_name, retriever_fn in retrievers.items():
        retrieved = retriever_fn(query, top_k=TOP_K)
        retrieved_ids = retrieved["doc_id"].tolist()

        evaluation_rows.append(
            {
                "query_id": record["query_id"],
                "retriever": retriever_name,
                "precision_at_3": precision_at_k(retrieved_ids, relevant_ids, TOP_K),
                "hit_rate_at_3": hit_rate_at_k(retrieved_ids, relevant_ids, TOP_K),
                "recall_at_3": recall_at_k(retrieved_ids, relevant_ids, TOP_K),
            }
        )

        for rank, doc_id in enumerate(retrieved_ids, start=1):
            ranking_rows.append(
                {
                    "query_id": record["query_id"],
                    "query": query,
                    "retriever": retriever_name,
                    "rank": rank,
                    "doc_id": doc_id,
                    "is_relevant": doc_id in relevant_ids,
                }
            )

metrics_df = pd.DataFrame(evaluation_rows)
rankings_df = pd.DataFrame(ranking_rows)

display(metrics_df)


In [ ]:
# Aggregate results by retriever.

summary_df = (
    metrics_df.groupby("retriever")[["precision_at_3", "hit_rate_at_3", "recall_at_3"]]
    .mean()
    .sort_values("precision_at_3", ascending=False)
    .reset_index()
)

display(summary_df)


In [ ]:
# Inspect ranking failures.
# These are often more informative than the aggregate numbers.

failure_cases = rankings_df[(rankings_df["rank"] == 1) & (~rankings_df["is_relevant"])]
failure_cases


## Exercises

- Change the query wording so lexical overlap becomes weaker. Which retriever degrades first?
- Add a more culturally specific query and note whether dense retrieval helps.
- Change the chunk size policy in a real dataset and discuss how it could affect precision and recall.
- Identify cases where a fluent generator might hide the fact that the retriever failed.
